# Notebook 1: RAG and the decision record

**Outcome (Area 1):** a RAG agent that answers with cited sources, is scored on the RAG triad, and comes with a **decision record** that explains *why* its settings were chosen, plus a **SKILL.md** a peer can use without asking you anything.

## The one idea to explain
> "I didn't just build RAG. I tried a few settings, measured them, picked one by a written rule, and recorded why."

**What is a decision record?** A one-page memo saved next to the code: *the decision, the options tried, the evidence, the rule used to pick, the trade-offs, and when to revisit.* Anyone can read it later and challenge it with numbers instead of opinions.

## What this notebook does (in order)
1. A small knowledge base and a labelled question set (8 in-domain, 2 out-of-domain).
2. Retrieval: split text into chunks, rank chunks by TF-IDF similarity. *(Swap point for real embeddings: the `Index` class.)*
3. The agent: retrieve, answer **only** from context, cite `[n]`, say "I don't know" otherwise.
4. **Experiment grid:** 3 chunk sizes x 2 values of k = 6 settings, each measured.
5. **Decision record**, generated from the table by a written rule.
6. RAG triad scores (context relevance, groundedness, answer relevance) for the chosen setting.
7. **SKILL.md** generated from the same facts.
8. Tests that check all of the above.

Model switch: `BACKEND = "mock"` (default, no key) or `"real"` (Gemini). Only `ask_model()` talks to a model.

## Cell 1: Setup (model switch)

In [2]:
%pip install -q openai
import json, os, re, time, math, copy, statistics
from collections import Counter

BACKEND  = "mock"      # "mock" = built-in fake model (no key needed) | "real" = Gemini (key in Colab Secrets)
BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
MODELS   = ["gemini-2.5-flash-lite", "gemini-2.5-flash", "gemini-flash-lite-latest"]   # tried in order if one is busy

def get_key():
    for name in ("GEMINI_API_KEY", "GOOGLE_API_KEY"):          # Colab Secrets first, then environment variable
        try:
            from google.colab import userdata
            return userdata.get(name)
        except Exception:
            if os.environ.get(name): return os.environ[name]

_CACHE = {}
def ask_model(prompt):
    """The ONLY place a model is called: mock = a local function, real = Gemini."""
    if BACKEND == "mock":
        return mock_model(prompt)
    if prompt in _CACHE:
        return _CACHE[prompt]
    key = get_key()
    if not key:
        raise RuntimeError("BACKEND is 'real' but no key found: add GEMINI_API_KEY in Colab Secrets, or set BACKEND = 'mock'.")
    from openai import OpenAI
    client = OpenAI(base_url=BASE_URL, api_key=key, timeout=45, max_retries=0)
    last = None
    for round_ in range(3):
        for model in MODELS:
            try:
                r = client.chat.completions.create(model=model, temperature=0, messages=[{"role": "user", "content": prompt}])
                _CACHE[prompt] = r.choices[0].message.content.strip()
                return _CACHE[prompt]
            except Exception as e:
                last = e
                if type(e).__name__ not in ("InternalServerError", "RateLimitError", "APITimeoutError", "APIConnectionError", "ServiceUnavailableError"):
                    raise                      # a real mistake (bad key, wrong model name): stop and show it
        time.sleep(5 * (round_ + 1))
    raise RuntimeError(f"All models busy after retries: {last}")


# ---------- the fake model (used when BACKEND = "mock"): small rules that behave like a sensible model ----------
DECLINE_TEXT = "I don't know based on the available sources."
STOP = set("the a an of to in is it and or for on with as by be are that this at from can how what why which does do who when where than then so but not you your".split())
def toks(text):
    out = []
    for t in re.findall(r"[a-z0-9]+", text.lower()):
        if t in STOP: continue
        if len(t) > 3 and t.endswith("s"): t = t[:-1]       # tiny stemmer: "chunks" -> "chunk"
        out.append(t)
    return out

def sentences(text):
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", text) if s.strip()]

def _after(prompt, tag):
    return prompt.split(tag, 1)[1].strip()

def mock_model(prompt):
    task = prompt.split("\n", 1)[0].strip()
    if task == "TASK:ANSWER":
        ctx = re.findall(r"^\[(\d+)\] \(source: [^)]*\) (.*)$", prompt, re.M)
        qt = set(toks(_after(prompt, "QUESTION:")))
        cand = [(len(qt & set(toks(s))), n, s) for n, text in ctx for s in sentences(text)]
        good = [c for c in sorted(cand, key=lambda c: -c[0]) if c[0] >= 2][:2]    # needs >= 2 shared words, else decline
        return " ".join(f"{s} [{n}]" for _, n, s in good) if good else DECLINE_TEXT
    if task == "TASK:RELEVANCE":
        q = set(toks(prompt.split("QUESTION:", 1)[1].split("PASSAGE:", 1)[0])); p = set(toks(_after(prompt, "PASSAGE:")))
        return str(round(10 * len(q & p) / max(len(q), 1)))
    if task == "TASK:GROUNDED":
        ctx = set(toks(prompt.split("CONTEXT:", 1)[1].split("STATEMENT:", 1)[0])); st = toks(prompt.split("STATEMENT:", 1)[1].split("Is the STATEMENT", 1)[0])
        return "YES" if st and sum(t in ctx for t in st) / len(st) >= 0.7 else "NO"
    if task == "TASK:ANSWER_REL":
        q = set(toks(prompt.split("QUESTION:", 1)[1].split("ANSWER:", 1)[0])); a = set(toks(_after(prompt, "ANSWER:")))
        return str(min(10, round(10 * len(q & a) / max(len(q), 1) + 2)))
    return "OK"

## Cell 2: Knowledge base and labelled question set
`DOCS` is the corpus (swap for your own). `EVAL` is the labelled set: **question, expected source, a key phrase the answer must contain**. Out-of-domain questions have no source: the correct behaviour is to decline.

In [3]:
DOCS = {
 "chunking_guide": "Chunking splits documents into pieces small enough to embed and retrieve. Fixed-size chunks are simple but can cut ideas in half. Overlap of 10-20% between chunks preserves context across boundaries. Semantic chunking splits on topic shifts and usually gives more coherent chunks at higher cost.",
 "vector_stores": "A vector store indexes embeddings so that nearest-neighbour search is fast. Chroma is a lightweight open-source option that runs in-process, which makes it popular for prototypes. Cosine similarity is the most common distance metric for text embeddings. Metadata filters let you restrict retrieval to a source, date, or tenant.",
 "embeddings": "Embeddings map text to dense vectors so that semantically similar texts land close together. Separate query and document modes usually improve retrieval quality. Visualising embeddings with t-SNE or UMAP helps spot clusters and outliers. These plots are for intuition rather than measurement.",
 "rag_evaluation": "The RAG triad evaluates three links: context relevance, groundedness and answer relevance. Context relevance asks whether retrieval returned useful chunks for the question. Groundedness asks whether every claim in the answer is supported by the retrieved context. Answer relevance asks whether the answer addresses the question asked. LLM-as-a-judge scoring needs no labels but should be run deterministically and spot-checked by a human.",
 "advanced_rag": "Query rewriting turns a vague or conversational question into a better search query before retrieval. Multi-query expansion retrieves for several paraphrases and merges the results. Re-ranking reorders the top 20-50 retrieved chunks with a stronger cross-encoder or LLM and keeps only the best few. Re-ranking improves precision at the cost of latency.",
 "failure_modes": "Hallucination in RAG usually appears when retrieval misses and the model fills the gap from its own memory. Instructing the model to say it does not know when context is insufficient reduces this. Citation errors occur when the model attributes a claim to the wrong chunk. Numbering chunks in the prompt and asking for bracketed citations makes verification straightforward.",
}

# (question, expected source, key phrase that a correct answer contains)
EVAL = [
 ("What is the difference between fixed-size and semantic chunking?", "chunking_guide", "topic shifts"),
 ("Which distance metric is most common for text embeddings?", "vector_stores", "cosine"),
 ("What are the three parts of the RAG triad?", "rag_evaluation", "context relevance"),
 ("Why does hallucination happen in RAG?", "failure_modes", "retrieval misses"),
 ("How does query rewriting help retrieval?", "advanced_rag", "better search query"),
 ("What does re-ranking improve and at what cost?", "advanced_rag", "latency"),
 ("How do bracketed citations help?", "failure_modes", "verification"),
 ("What does groundedness ask about every claim in the answer?", "rag_evaluation", "supported by the retrieved context"),
]
OUT_OF_DOMAIN = ["What is the capital of Australia?", "Who won the 2018 football world cup?"]
print(len(DOCS), "documents,", len(EVAL), "in-domain questions,", len(OUT_OF_DOMAIN), "out-of-domain questions")

6 documents, 8 in-domain questions, 2 out-of-domain questions


## Cell 3: Chunking and retrieval
- `make_chunks(size)`: groups `size` sentences per chunk. **Chunk size is decision #1.**
- `Index.search(q, k)`: returns the top `k` chunks. **k is decision #2.**
- Ranking is TF-IDF cosine (pure Python, no downloads). To use embeddings, replace `Index` with one that calls an embedding model; nothing else changes.

In [4]:
def make_chunks(size):
    chunks = []
    for src, text in DOCS.items():
        ss = sentences(text)
        for i in range(0, len(ss), size):
            chunks.append({"id": f"{src}-{i // size}", "source": src, "text": " ".join(ss[i:i + size])})
    return chunks

class Index:
    def __init__(self, chunks):
        self.chunks = chunks
        tfs = [Counter(toks(c["text"])) for c in chunks]
        df = Counter(t for tf in tfs for t in tf)
        n = len(chunks)
        self.idf = {t: math.log((n + 1) / (df[t] + 0.5)) for t in df}
        self.vecs = [self._vec(tf) for tf in tfs]
    def _vec(self, tf):
        v = {t: c * self.idf.get(t, 0) for t, c in tf.items()}
        norm = math.sqrt(sum(x * x for x in v.values())) or 1
        return {t: x / norm for t, x in v.items()}
    def search(self, q, k):
        qv = self._vec(Counter(toks(q)))
        scored = sorted(((sum(w * v.get(t, 0) for t, w in qv.items()), i) for i, v in enumerate(self.vecs)), key=lambda x: (-x[0], x[1]))
        return [self.chunks[i] for s, i in scored[:k] if s > 0]

idx = Index(make_chunks(2))
for c in idx.search("Which distance metric is most common for text embeddings?", 2):
    print(c["id"], "|", c["text"][:90], "...")

vector_stores-1 | Cosine similarity is the most common distance metric for text embeddings. Metadata filters ...
embeddings-0 | Embeddings map text to dense vectors so that semantically similar texts land close togethe ...


## Cell 4: The agent (retrieve, answer, cite)
The prompt forces three behaviours: use **only** the numbered context, **cite** every claim as `[n]`, and say **"I don't know"** if the context does not contain the answer. `cited_ids` and `sources` are read back from the answer, so a reviewer can verify each claim.

In [ ]:
PROMPT = """TASK:ANSWER
Answer ONLY using the numbered context. Cite every claim like [1]. If the context does not contain the answer, reply exactly: I don't know based on the available sources. Be concise (1-3 sentences).

CONTEXT:
{ctx}

QUESTION: {q}"""

def rag_answer(q, index, k):
    ctx = index.search(q, k)
    ctx_str = "\n".join(f"[{i + 1}] (source: {c['source']}) {c['text']}" for i, c in enumerate(ctx))
    ans = ask_model(PROMPT.format(ctx=ctx_str, q=q)) if ctx else DECLINE_TEXT
    cited = sorted({int(n) for n in re.findall(r"\[(\d+)\]", ans) if 0 < int(n) <= len(ctx)})
    return {"question": q, "answer": ans, "context": ctx,
            "cited_ids": [ctx[i - 1]["id"] for i in cited],
            "sources": sorted({ctx[i - 1]["source"] for i in cited})}

o = rag_answer("Which distance metric is most common for text embeddings?", idx, 2)
print(o["answer"]); print("cited:", o["cited_ids"], "| sources:", o["sources"])
print(rag_answer(OUT_OF_DOMAIN[0], idx, 2)["answer"])

## Cell 5: The experiment grid (this is the evidence)
Every setting is run on the same labelled set. Metrics:
- **hit_rate**: did retrieval bring back the expected source?
- **correct_rate**: does the answer contain the key phrase?
- **decline_ok**: did the agent decline both out-of-domain questions? (safety)
- **avg_ctx_chars**: how much text goes into the prompt (a proxy for cost and latency)

In [ ]:
import pandas as pd
GRID = [(size, k) for size in (1, 2, 4) for k in (1, 3)]

def run_config(size, k):
    index = Index(make_chunks(size)); hits = correct = ctx_chars = 0
    for q, src, key in EVAL:
        o = rag_answer(q, index, k)
        hits += src in [c["source"] for c in o["context"]]
        correct += key in o["answer"].lower()
        ctx_chars += sum(len(c["text"]) for c in o["context"])
    declined = sum(DECLINE_TEXT in rag_answer(q, index, k)["answer"] for q in OUT_OF_DOMAIN)
    n = len(EVAL)
    return {"chunk_size": size, "k": k, "hit_rate": round(hits / n, 2), "correct_rate": round(correct / n, 2),
            "decline_ok": round(declined / len(OUT_OF_DOMAIN), 2), "avg_ctx_chars": round(ctx_chars / n)}

TABLE = pd.DataFrame([run_config(s, k) for s, k in GRID])
TABLE

## Cell 6: Choose by a written rule, then write the decision record
**The rule (written before looking at the winner):**
1. Must decline every out-of-domain question (`decline_ok` = 1.0). Safety first.
2. Then highest `correct_rate`.
3. Then highest `hit_rate`.
4. Then smallest `avg_ctx_chars` (cheaper, faster).

The record is **generated from the table**, so its numbers cannot drift from the evidence.

In [ ]:
def choose(table):
    ranked = table.sort_values(["decline_ok", "correct_rate", "hit_rate", "avg_ctx_chars"], ascending=[False, False, False, True])
    return ranked.iloc[0].to_dict(), ranked

def md_table(df):
    cols = list(df.columns)
    rows = ["| " + " | ".join(cols) + " |", "|" + "---|" * len(cols)]
    rows += ["| " + " | ".join(str(r[c]) for c in cols) + " |" for r in df.to_dict("records")]
    return "\n".join(rows)

def decision_record(table, best, backend):
    return f"""# Decision record: RAG retrieval settings

- **Date:** {time.strftime("%Y-%m-%d")}
- **Status:** accepted
- **Backend used for the experiment:** {backend}

## Decision
Split documents into chunks of **{int(best['chunk_size'])} sentence(s)** and retrieve the top **{int(best['k'])}** chunks per question.

## Context
The agent must answer from our documents with citations and must refuse when the documents do not contain the answer. Chunk size and k are the two settings that most affect that, so we measured them instead of guessing.

## Options considered (same {len(EVAL)} in-domain + {len(OUT_OF_DOMAIN)} out-of-domain questions each)
{md_table(table)}

## Why this option won
Selection rule, fixed in advance: (1) decline every out-of-domain question, (2) highest correct_rate, (3) highest hit_rate, (4) smallest avg_ctx_chars.
Winner: chunk_size={int(best['chunk_size'])}, k={int(best['k'])} with correct_rate={best['correct_rate']}, hit_rate={best['hit_rate']}, decline_ok={best['decline_ok']}, avg_ctx_chars={int(best['avg_ctx_chars'])}.

## Trade-offs
- More chunks (higher k) can raise hit_rate but adds prompt text, which costs latency and money.
- Very large chunks dilute ranking; very small chunks can split an answer across two chunks.

## Limits of the evidence
- Only {len(EVAL) + len(OUT_OF_DOMAIN)} questions: one question changes a rate by {round(100 / len(EVAL))} points. Treat small gaps as ties.
- Retrieval here is TF-IDF; results with real embeddings may differ.

## Revisit when
- The document set changes or grows a lot.
- The labelled set grows past ~50 questions.
- groundedness or answer relevance drops in a later run.
- Retrieval is switched to embeddings or a re-ranker is added.
"""

BEST, RANKED = choose(TABLE)
RECORD = decision_record(TABLE, BEST, BACKEND)
open("decision_record.md", "w").write(RECORD)
print(RECORD)

## Cell 7: RAG triad scores for the chosen setting
Three judges, each asked for **one short answer** (a number or YES/NO) so parsing is simple:
- **Context relevance:** is each retrieved chunk useful for the question? (0-10, scaled to 0-1)
- **Groundedness:** is every sentence of the answer supported by the context? (share of YES)
- **Answer relevance:** does the answer address the question? (0-10, scaled to 0-1)

For a correct "I don't know" there are no claims, so groundedness is 1.0 and answer relevance is not applicable.

In [ ]:
def parse_int(text):
    m = re.search(r"\d+", text)
    return min(int(m.group()), 10) / 10 if m else 0.0

def context_relevance(q, ctx):
    if not ctx: return 0.0
    s = [parse_int(ask_model(f"TASK:RELEVANCE\nRate from 0 to 10 how useful the PASSAGE is for answering the QUESTION. Reply with ONLY one integer.\n\nQUESTION: {q}\n\nPASSAGE: {c['text']}")) for c in ctx]
    return sum(s) / len(s)

def groundedness(ans, ctx):
    ctx_str = "\n".join(f"[{i + 1}] {c['text']}" for i, c in enumerate(ctx))
    sents = [re.sub(r"\[\d+\]", "", s).strip() for s in sentences(ans)]
    sents = [s for s in sents if len(s) > 15 and DECLINE_TEXT not in s]
    if not sents: return 1.0
    yes = sum(ask_model(f"TASK:GROUNDED\nCONTEXT:\n{ctx_str}\n\nSTATEMENT: {s}\n\nIs the STATEMENT fully supported by the CONTEXT? Reply with ONLY YES or NO.").strip().upper().startswith("YES") for s in sents)
    return yes / len(sents)

def answer_relevance(q, ans):
    return parse_int(ask_model(f"TASK:ANSWER_REL\nRate from 0 to 10 how directly the ANSWER addresses the QUESTION (ignore whether it is correct). Reply with ONLY one integer.\n\nQUESTION: {q}\n\nANSWER: {ans}"))

def triad(q, index, k):
    o = rag_answer(q, index, k)
    declined = DECLINE_TEXT in o["answer"]
    return {"question": q, "context_relevance": round(context_relevance(q, o["context"]), 2),
            "groundedness": round(groundedness(o["answer"], o["context"]), 2),
            "answer_relevance": None if declined else round(answer_relevance(q, o["answer"]), 2),
            "declined": declined, "sources": ", ".join(o["sources"])}

BEST_INDEX = Index(make_chunks(int(BEST["chunk_size"])))
TRIAD = pd.DataFrame([triad(q, BEST_INDEX, int(BEST["k"])) for q in [e[0] for e in EVAL] + OUT_OF_DOMAIN])
print(TRIAD[["context_relevance", "groundedness", "answer_relevance"]].mean().round(2))
TRIAD

## Cell 8: SKILL.md (hand this to a peer who has not seen the agent)

In [ ]:
def make_skill_md(best, triad_means):
    return f"""---
name: cited-rag-agent
description: Answers questions from a document set with numbered citations, refuses when the documents do not contain the answer, and scores each answer on the RAG triad. Use when you need a source-backed Q&A agent over your own documents.
---

# Cited RAG agent

## When to use
- You have documents and need answers that cite sources.
- You want a repeatable quality score, not a gut check.

## Inputs
- `DOCS`: dict of source_name -> text.
- `EVAL`: list of (question, expected_source, key_phrase). Add 1-2 out-of-domain questions that must be declined.

## Steps
1. Run the experiment grid (chunk sizes 1/2/4 sentences x k 1/3) on your labelled set.
2. Pick by the rule in the decision record: decline all out-of-domain, then correct_rate, then hit_rate, then smaller context.
3. Use the chosen setting: chunk_size={int(best['chunk_size'])}, k={int(best['k'])} (from our documents; re-run on yours).
4. Answer with numbered context, require `[n]` citations, require "I don't know" when context is insufficient.
5. Score with the triad and read the lowest rows by hand.

## Output
- Answer text with `[n]` citations plus source names.
- `decision_record.md` explaining the settings.

## Reading the triad (our run: context relevance {triad_means['context_relevance']}, groundedness {triad_means['groundedness']}, answer relevance {triad_means['answer_relevance']})
- Low context relevance: fix chunking or k; consider query rewriting or re-ranking.
- Low groundedness: tighten the prompt; the model is using outside knowledge.
- Low answer relevance: the question is ambiguous or retrieval missed.

## Known limits
- Model judges are noisy: spot-check at least 10% of rows by hand.
- Retrieval is TF-IDF in this notebook; swap `Index` for embeddings when meaning matters more than shared words.
- Tiny labelled set: treat small differences as ties.
"""

MEANS = TRIAD[["context_relevance", "groundedness", "answer_relevance"]].mean().round(2).to_dict()
SKILL = make_skill_md(BEST, MEANS)
open("SKILL.md", "w").write(SKILL)
print(SKILL[:700], "...")

## Cell 9: Tests
Each test names what it proves. In `mock` mode all must pass. In `real` mode treat failures as findings about the model, not as bugs.

In [ ]:
RESULTS = []
def check(name, cond, detail=""):
    RESULTS.append((name, bool(cond)))
    print(("PASS" if cond else "FAIL"), "-", name, ("" if cond else detail))

# 1. chunking keeps every sentence
for size in (1, 2, 4):
    total = sum(len(sentences(t)) for t in DOCS.values())
    kept = sum(len(sentences(c["text"])) for c in make_chunks(size))
    check(f"chunking size={size} keeps every sentence", kept == total, f"{kept} vs {total}")
check("size-2 chunking makes fewer chunks than size-1", len(make_chunks(2)) < len(make_chunks(1)))

# 2. retrieval finds the right document
check("retrieval: distance-metric question hits vector_stores", "vector_stores" in [c["source"] for c in idx.search(EVAL[1][0], 3)])

# 3. answers are cited and citations are real
o = rag_answer(EVAL[2][0], BEST_INDEX, int(BEST["k"]))
check("answer has at least one [n] citation", len(o["cited_ids"]) >= 1, o["answer"])
check("cited chunk ids exist in the retrieved context", all(i in [c["id"] for c in o["context"]] for i in o["cited_ids"]))
check("sources come from the cited chunks only", set(o["sources"]) <= {c["source"] for c in o["context"]})

# 4. out-of-domain questions are declined
for q in OUT_OF_DOMAIN:
    check(f"declines out-of-domain: {q}", DECLINE_TEXT in rag_answer(q, BEST_INDEX, int(BEST["k"]))["answer"])

# 5. judge parsing and judge behaviour
check("parse_int reads 'Score: 7' as 0.7", parse_int("Score: 7") == 0.7)
check("parse_int caps 15 at 1.0 and defaults to 0.0", parse_int("15") == 1.0 and parse_int("none") == 0.0)
good_ctx = idx.search(EVAL[1][0], 2)
good = "Cosine similarity is the most common distance metric for text embeddings."
bad = "The Eiffel Tower in Paris was completed in 1889 and attracts millions of visitors."
check("groundedness: supported answer scores above unsupported answer", groundedness(good, good_ctx) > groundedness(bad, good_ctx), f"{groundedness(good, good_ctx)} vs {groundedness(bad, good_ctx)}")
check("answer relevance: on-topic answer scores above off-topic answer", answer_relevance(EVAL[1][0], good) > answer_relevance(EVAL[1][0], bad))
check("context relevance: relevant chunks score above irrelevant chunks", context_relevance(EVAL[1][0], good_ctx) > context_relevance(EVAL[1][0], idx.search("Overlap between chunks preserves context", 1)))

# 6. the decision record matches the evidence
top = TABLE.sort_values(["decline_ok", "correct_rate", "hit_rate", "avg_ctx_chars"], ascending=[False, False, False, True]).iloc[0]
check("chosen setting follows the written rule", (BEST["chunk_size"], BEST["k"]) == (top["chunk_size"], top["k"]))
check("chosen setting declines every out-of-domain question", BEST["decline_ok"] == 1.0)
check("record names the chosen chunk size and k", f"**{int(BEST['chunk_size'])} sentence(s)**" in RECORD and f"top **{int(BEST['k'])}**" in RECORD)
check("record contains every option's row (all 6 settings)", all(f"| {r.chunk_size} | {r.k} |" in RECORD for r in TABLE.itertuples()))
check("record has the required sections", all(h in RECORD for h in ("## Decision", "## Options considered", "## Why this option won", "## Trade-offs", "## Revisit when")))

# 7. SKILL.md is usable
check("SKILL.md has name and description front matter", SKILL.startswith("---\nname: cited-rag-agent\ndescription:"))
check("SKILL.md has Inputs, Steps, Output, Known limits", all(h in SKILL for h in ("## Inputs", "## Steps", "## Output", "## Known limits")))
check("files were written", os.path.exists("decision_record.md") and os.path.exists("SKILL.md"))

print(f"\n{sum(ok for _, ok in RESULTS)}/{len(RESULTS)} tests passed")
assert all(ok for _, ok in RESULTS), "some tests failed (in real mode this is a finding, not necessarily a bug)"

## How to explain this in 2 minutes
1. **Problem:** a RAG agent has settings (chunk size, k). Most people guess them.
2. **What I did:** tried 6 settings on a labelled set, measured hit rate, correctness, safety (declines) and cost.
3. **Decision:** chose by a rule written in advance; the record is generated from the table, so claims match evidence.
4. **Quality:** RAG triad scores on the chosen setting, with judges that have their own tests.
5. **Handover:** SKILL.md so a peer can run it on their own documents.

## Questions a reviewer may ask
| Question | Answer |
|---|---|
| Why that chunk size and k? | The decision record: the rule, the table, the winner. |
| How do you know the judge is not broken? | Tests 5: it scores a supported answer above an unsupported one, and an on-topic answer above an off-topic one. |
| What if the model makes up an answer? | Prompt forces citations and "I don't know"; out-of-domain questions are in the labelled set and gate the choice. |
| What would change the decision? | The "Revisit when" section: new documents, bigger eval set, falling triad scores, embeddings or re-ranking. |
| Is the sample big enough? | No, 10 questions. The record says so; small gaps are treated as ties. |
| How do I use a real model? | Set `BACKEND = "real"`; only `ask_model()` changes. Retrieval can move to embeddings by replacing `Index`. |
| Where is quantization? | Not used here: the model is called through an API. Quantization applies when you host an open model yourself (smaller weights, less memory, small quality cost). |